# Parte 1 — Meio de transporte por cor/raça (Brasil)

Análise da Tabela 13 (PNAD Contínua / IBGE): meio de transporte que as mulheres ocupadas passam mais tempo no deslocamento ao trabalho, por cor ou raça — Brasil, 2022.

## 1. Importando as bibliotecas

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')


## 2. Lendo o arquivo Excel (aba 'Exemplo gráfico Brasil')

In [ ]:

arquivo = 'Tabela_13_Meio_de_transporte.xlsx'



df = pd.read_excel(arquivo, sheet_name='Exemplo gráfico Brasil', skiprows=1)


df = df.rename(columns={'Meio de transporte/cor ou raça': 'Meio de Transporte'})

df


## 3. Gráfico de pizza (pie chart) por cor/raça

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
cores = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0', '#ffb3e6']

grupos = ['Branca', 'Preta ou parda', 'Indígena']

for i, col in enumerate(grupos):
    axes[i].pie(
        df[col],
        labels=df['Meio de Transporte'],
        autopct='%1.3f%%',  
        startangle=180,      
        colors=cores,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1.2},
    )
    axes[i].set_title(f'Cor/Raça: {col}', fontsize=12, fontweight='bold')

plt.suptitle(
    'Meio de transporte utilizado por mulheres no deslocamento ao trabalho, por cor ou raça — Brasil, 2022',
    fontsize=14,
    fontweight='bold',
)
plt.tight_layout()
plt.savefig('grafico_transporte.png', dpi=300)
plt.show()


## 4. Calculando as diferenças entre os grupos (para a análise abaixo)

In [ ]:
diferencas = df.set_index('Meio de Transporte')[grupos].copy()

diferencas['Branca - Preta ou parda'] = diferencas['Branca'] - diferencas['Preta ou parda']
diferencas['Branca - Indígena'] = diferencas['Branca'] - diferencas['Indígena']
diferencas['Preta ou parda - Indígena'] = diferencas['Preta ou parda'] - diferencas['Indígena']

diferencas.round(1)


## 5. Análise dos gráficos

**Diferença entre os meios de transporte por cor/raça (mulheres, Brasil, 2022):**

- **Automóvel, táxi ou assemelhados**
  - Mulheres brancas − Mulheres pretas ou pardas = 41,8% − 20,6% = **21,2 p.p.** (brancas usam muito mais carro)
  - Mulheres brancas − Mulheres indígenas = 41,8% − 15,2% = **26,6 p.p.**
  - Mulheres pretas ou pardas − Mulheres indígenas = 20,6% − 15,2% = **5,4 p.p.**

- **Transporte coletivo**
  - Mulheres brancas − Mulheres pretas ou pardas = 25,2% − 34,6% = **-9,4 p.p.** (pretas/pardas usam mais transporte coletivo)
  - Mulheres brancas − Mulheres indígenas = 25,2% − 22,4% = **2,8 p.p.**
  - Mulheres pretas ou pardas − Mulheres indígenas = 34,6% − 22,4% = **12,2 p.p.**

- **A pé**
  - Mulheres brancas − Mulheres pretas ou pardas = 19,4% − 24,8% = **-5,4 p.p.**
  - Mulheres brancas − Mulheres indígenas = 19,4% − 37,5% = **-18,1 p.p.** (indígenas se deslocam muito mais a pé)
  - Mulheres pretas ou pardas − Mulheres indígenas = 24,8% − 37,5% = **-12,7 p.p.**

- **Motocicleta ou mototáxi**
  - Mulheres brancas − Mulheres pretas ou pardas = 9,9% − 14,2% = **-4,3 p.p.**
  - Mulheres brancas − Mulheres indígenas = 9,9% − 15,4% = **-5,5 p.p.**
  - Mulheres pretas ou pardas − Mulheres indígenas = 14,2% − 15,4% = **-1,2 p.p.**

- **Bicicleta**
  - Mulheres brancas − Mulheres pretas ou pardas = 3,2% − 5,2% = **-2,0 p.p.**
  - Mulheres brancas − Mulheres indígenas = 3,2% − 5,9% = **-2,7 p.p.**
  - Mulheres pretas ou pardas − Mulheres indígenas = 5,2% − 5,9% = **-0,7 p.p.**

- **Outros**
  - Diferenças pequenas entre brancas e pretas/pardas (0,5% vs 0,6%), mas mulheres indígenas se destacam com 3,6%, bem acima dos outros dois grupos.

**Conclusão geral:** mulheres brancas concentram-se muito mais no uso de automóvel/táxi, enquanto mulheres pretas ou pardas dependem proporcionalmente mais do transporte coletivo, e mulheres indígenas se deslocam muito mais a pé e usam mais moto/mototáxi do que os demais grupos. Isso reflete, entre outros fatores, diferenças de renda e de acesso a veículo próprio entre os grupos de cor/raça.

# Parte 2 — Meio de transporte por Brasil, Grande Região, UF e Município

## 1. Lendo o arquivo Excel (aba 'BR GR UF MU')

In [ ]:

raw = pd.read_excel(arquivo, sheet_name='BR GR UF MU', header=None)

linha_grupos = raw.iloc[1].ffill()   
linha_modos = raw.iloc[2]           

colunas = ['Local']
for i in range(1, raw.shape[1]):
    colunas.append(f'{linha_grupos[i]} - {linha_modos[i]}')

dados = raw.iloc[3:].copy()
dados.columns = colunas
dados = dados.reset_index(drop=True)


for c in colunas[1:]:
    dados[c] = pd.to_numeric(dados[c].replace('.', pd.NA), errors='coerce')

dados.head()


A tabela concatena, na mesma coluna `Local`, o Brasil, as 5 Grandes Regiões, as 27 Unidades da Federação e todos os municípios (nesta ordem). Vamos separar cada bloco:

In [ ]:
brasil = dados.iloc[0:1].reset_index(drop=True)
regioes = dados.iloc[1:6].reset_index(drop=True)
estados = dados.iloc[6:33].reset_index(drop=True)    
municipios = dados.iloc[33:].reset_index(drop=True)

print('Brasil:', len(brasil), 'linha')
print('Regiões:', len(regioes), 'linhas')
print('Estados/UF:', len(estados), 'linhas')
print('Municípios:', len(municipios), 'linhas')


## 2. Listando apenas os estados

In [ ]:
lista_estados = estados['Local'].tolist()
for uf in lista_estados:
    print(uf)


## 3. Respondendo às questões

> **Observação:** a planilha não traz uma coluna "Total" combinando as três cores/raças — só `Branca`, `Preta ou Parda` e `Indígena` separadamente. Por isso, para responder o que é pedido "em geral", calculamos a **média simples entre os três grupos de cor/raça** em cada linha (estado ou município) e comparamos essa média entre os locais.

### Estado com MAIS mulheres, em geral, usando transporte coletivo

In [ ]:
cols_coletivo = [c for c in estados.columns if 'Transporte coletivo' in c]
estados['Média - Transporte coletivo'] = estados[cols_coletivo].mean(axis=1)

estado_mais_coletivo = estados.loc[estados['Média - Transporte coletivo'].idxmax()]
print(f"Estado com MAIS mulheres usando transporte coletivo: {estado_mais_coletivo['Local']} "
      f"({estado_mais_coletivo['Média - Transporte coletivo']:.2f}% em média entre as cores/raças)")

estados[['Local', 'Média - Transporte coletivo']].sort_values('Média - Transporte coletivo', ascending=False).head(5)


### Estado com MENOS mulheres, em geral, usando transporte coletivo

In [ ]:
estado_menos_coletivo = estados.loc[estados['Média - Transporte coletivo'].idxmin()]
print(f"Estado com MENOS mulheres usando transporte coletivo: {estado_menos_coletivo['Local']} "
      f"({estado_menos_coletivo['Média - Transporte coletivo']:.2f}% em média entre as cores/raças)")

estados[['Local', 'Média - Transporte coletivo']].sort_values('Média - Transporte coletivo').head(5)


### Cidade do Brasil com MAIS mulheres andando de bicicleta

In [ ]:
cols_bike = [c for c in municipios.columns if 'Bicicleta' in c]
municipios['Média - Bicicleta'] = municipios[cols_bike].mean(axis=1)

cidade_mais_bike = municipios.loc[municipios['Média - Bicicleta'].idxmax()]
print(f"Cidade com MAIS mulheres andando de bicicleta: {cidade_mais_bike['Local']} "
      f"({cidade_mais_bike['Média - Bicicleta']:.2f}% em média entre as cores/raças)")

municipios[['Local', 'Média - Bicicleta']].sort_values('Média - Bicicleta', ascending=False).head(10)


### Cidade do Brasil com MAIS mulheres utilizando carro (automóvel/táxi)

In [ ]:
cols_carro = [c for c in municipios.columns if 'Automóvel' in c]
municipios['Média - Automóvel'] = municipios[cols_carro].mean(axis=1)

cidade_mais_carro = municipios.loc[municipios['Média - Automóvel'].idxmax()]
print(f"Cidade com MAIS mulheres usando carro/táxi: {cidade_mais_carro['Local']} "
      f"({cidade_mais_carro['Média - Automóvel']:.2f}% em média entre as cores/raças)")

municipios[['Local', 'Média - Automóvel']].sort_values('Média - Automóvel', ascending=False).head(10)


## Resumo das respostas

| Pergunta | Resposta |
|---|---|
| Estado com **mais** mulheres usando transporte coletivo | **Rio de Janeiro** (≈ 52,2% em média entre as cores/raças) |
| Estado com **menos** mulheres usando transporte coletivo | **Rondônia** (≈ 4,3% em média entre as cores/raças) |
| Cidade com **mais** mulheres andando de bicicleta | **Tuiuti (SP)** (100% em média — atenção: cidade muito pequena, valor pouco representativo estatisticamente) |
| Cidade com **mais** mulheres usando carro/táxi | **Bom Jesus do Oeste (SC)** (≈ 86,1% em média) |

**Atenção:** em municípios muito pequenos, a amostra da PNAD Contínua é reduzida e alguns valores podem ficar em 100% ou próximos disso por causa do tamanho pequeno da amostra, não necessariamente porque a cidade inteira se comporta assim — vale olhar o ranking completo (`sort_values`), e não só o primeiro colocado, ao tirar conclusões.